# LifeMate — Qwen3.5-4B gold next-action SFT

This notebook trains a QLoRA adapter from the prepared train-only SFT JSONL. It does not duplicate the BTC competition dataset, submit to Kaggle, or change any default model.

## Attach these inputs in Kaggle

1. BTC competition data: `gemma-4-developer-agent` (the existing competition input).
2. Prepared SFT data: `daoviet/lifemate-qwen35-sft-data-v1` (upload the separate `kaggle-upload` folder as a private dataset).
3. Base model: `limamateus/qwen3-5-4b` (Transformers model input).

Target this notebook at Kaggle 2×T4: it selects FP16 on T4, uses 4-bit QLoRA, micro-batch 1, gradient checkpointing, SDPA, and gradient accumulation 8 per GPU (effective batch 16). Each GPU holds a full quantized model copy; the two GPUs do not pool VRAM. The 8,192-token cap can still be memory-heavy, so the notebook runs a forward/backward preflight on the longest example before training. If that preflight runs out of memory, lower MAX_SEQ_LENGTH to 6,144, then rerun the setup and tokenization cells. Internet can stay off; inputs are read locally.

Training supervises only the action JSON. The Qwen3.5 template is rendered in thinking mode; the loss ignores the think-close delimiter and all prompt tokens, so this run does not teach synthetic chain-of-thought or force a short reasoning budget.


In [ ]:
import json, os, random, subprocess, sys, time
from pathlib import Path
from importlib.metadata import version
import torch
from transformers import AutoTokenizer

SEED = 20261106
NUM_EPOCHS = 2
LEARNING_RATE = 5e-5
MAX_SEQ_LENGTH = 8192
EFFECTIVE_BATCH_TARGET = 16
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
INPUT_ROOT = Path('/kaggle/input')
OUTPUT_ROOT = Path('/kaggle/working/qwen35_lifemate_sft_v1')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def unique_path(paths, label):
    paths = sorted(set(Path(p).resolve() for p in paths))
    if len(paths) != 1:
        raise RuntimeError(f'Expected exactly one {label}, found {len(paths)}: {paths[:8]}')
    return paths[0]

sft_file = unique_path(INPUT_ROOT.rglob('sft_train.jsonl'), 'sft_train.jsonl')
SFT_DIR = sft_file.parent
VAL_INPUTS_FILE = SFT_DIR / 'val309_inputs.jsonl'
TRAIN_IDS_FILE = SFT_DIR / 'train_ids.json'
VAL_IDS_FILE = SFT_DIR / 'val309_ids.json'
if not all(p.is_file() for p in [VAL_INPUTS_FILE, TRAIN_IDS_FILE, VAL_IDS_FILE]):
    raise FileNotFoundError(f'SFT dataset bundle is incomplete: {SFT_DIR}')

train_json_paths = list(INPUT_ROOT.rglob('train.jsonl'))
btc_candidates = []
for p in train_json_paths:
    train_dir = p.parent
    if (train_dir / 'train_tools_catalog.json').is_file() and (train_dir / 'zaic_tools').is_dir():
        btc_candidates.append(train_dir.parent)
BTC_ROOT = unique_path(btc_candidates, 'BTC competition data root')
BTC_TRAIN_DIR = BTC_ROOT / 'train'

model_configs = []
for p in INPUT_ROOT.rglob('config.json'):
    path_text = str(p.parent).lower().replace('_', '-')
    try:
        cfg = json.loads(p.read_text(encoding='utf-8'))
    except Exception:
        continue
    arch = ' '.join(cfg.get('architectures', [])).lower().replace('_', '-')
    model_type = str(cfg.get('model_type', '')).lower().replace('_', '-')
    if 'qwen3-5' in path_text and '4b' in path_text and ('qwen3-5' in arch or 'qwen3-5' in model_type):
        model_configs.append(p)
MODEL_DIR = unique_path(model_configs, 'Qwen3.5-4B model config').parent

GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT < 1:
    raise RuntimeError('Kaggle GPU is not enabled. Select a GPU accelerator before running training.')
try:
    TRANSFORMERS_VERSION = version('transformers')
    PEFT_VERSION = version('peft')
    BNB_VERSION = version('bitsandbytes')
except Exception as exc:
    raise RuntimeError('This runtime needs Transformers 5.x, PEFT, Accelerate, and bitsandbytes available offline.') from exc

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['WANDB_DISABLED'] = 'true'
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print({'BTC_ROOT': str(BTC_ROOT), 'SFT_DIR': str(SFT_DIR), 'MODEL_DIR': str(MODEL_DIR), 'gpus': [{'name': torch.cuda.get_device_name(i), 'vram_gib': round(torch.cuda.get_device_properties(i).total_memory / 2**30, 1), 'bf16': torch.cuda.is_bf16_supported()} for i in range(GPU_COUNT)], 'transformers': TRANSFORMERS_VERSION, 'peft': PEFT_VERSION, 'bitsandbytes': BNB_VERSION})


In [ ]:
# Audit and tokenize; only SFT train rows are passed to Trainer.
with sft_file.open(encoding='utf-8') as f:
    SFT_ROWS = [json.loads(line) for line in f if line.strip()]
TRAIN_IDS = set(json.loads(TRAIN_IDS_FILE.read_text(encoding='utf-8')))
VAL_IDS = set(json.loads(VAL_IDS_FILE.read_text(encoding='utf-8')))
assert len(TRAIN_IDS) == 1236 and len(VAL_IDS) == 309 and not TRAIN_IDS & VAL_IDS
assert {r['sample_id'] for r in SFT_ROWS} <= TRAIN_IDS
assert all(len(r['messages']) == 2 and r['messages'][0]['role'] == 'system' and r['messages'][1]['role'] == 'user' for r in SFT_ROWS)
assert all(r['target'].get('action') in {'tool_call', 'stop'} for r in SFT_ROWS)

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

THINK_CLOSE = '</think>\n\n'
encoded_rows = []
truncated_count = 0
lengths = []
for row in SFT_ROWS:
    prompt = tokenizer.apply_chat_template(row['messages'], tokenize=False, add_generation_prompt=True, enable_thinking=True)
    prompt_ids = tokenizer.encode(prompt, add_special_tokens=False)
    close_ids = tokenizer.encode(THINK_CLOSE, add_special_tokens=False)
    answer_text = json.dumps(row['target'], ensure_ascii=False, separators=(',', ':'))
    answer_ids = tokenizer.encode(answer_text, add_special_tokens=False) + [tokenizer.eos_token_id]
    room_for_prompt = MAX_SEQ_LENGTH - len(close_ids) - len(answer_ids)
    if room_for_prompt < 1:
        raise ValueError(f'Target exceeds MAX_SEQ_LENGTH: {row["sample_id"]}/{row["step_index"]}')
    if len(prompt_ids) > room_for_prompt:
        truncated_count += 1
        keep_head = int(room_for_prompt * 0.72)
        prompt_ids = prompt_ids[:keep_head] + prompt_ids[-(room_for_prompt - keep_head):]
    input_ids = prompt_ids + close_ids + answer_ids
    labels = [-100] * (len(prompt_ids) + len(close_ids)) + answer_ids
    encoded_rows.append({'input_ids': input_ids, 'labels': labels})
    lengths.append(len(input_ids))

TOKENIZED_PATH = Path('/tmp/lifemate_qwen35_sft_tokens.pt')
torch.save(encoded_rows, TOKENIZED_PATH)
print({'train_trajectories': len(TRAIN_IDS), 'val309_excluded_from_training': len(VAL_IDS), 'sft_rows': len(SFT_ROWS), 'target_action_counts': {}, 'sequence_p50': sorted(lengths)[len(lengths)//2], 'sequence_p95': sorted(lengths)[int(0.95*(len(lengths)-1))], 'sequence_max': max(lengths), 'truncated_prompts': truncated_count, 'truncated_pct': round(100*truncated_count/len(SFT_ROWS), 2), 'token_file_mb': round(TOKENIZED_PATH.stat().st_size/2**20, 1)})


In [ ]:
# One process per visible GPU. Each worker loads the same quantized base on its local GPU.
WORKER_SCRIPT = r'''
import argparse, json, os, random, sys
from pathlib import Path
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForMultimodalLM, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--model', required=True); ap.add_argument('--tokens', required=True); ap.add_argument('--out', required=True)
    ap.add_argument('--epochs', type=float, required=True); ap.add_argument('--lr', type=float, required=True)
    ap.add_argument('--max-length', type=int, required=True); ap.add_argument('--grad-accum', type=int, required=True)
    ap.add_argument('--seed', type=int, required=True); args = ap.parse_args()
    os.environ.setdefault('HF_HUB_OFFLINE','1'); os.environ.setdefault('TRANSFORMERS_OFFLINE','1'); os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
    rank=int(os.environ.get('LOCAL_RANK','0')); world=int(os.environ.get('WORLD_SIZE','1')); torch.cuda.set_device(rank)
    random.seed(args.seed); torch.manual_seed(args.seed); torch.cuda.manual_seed_all(args.seed)
    dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    tokenizer=AutoTokenizer.from_pretrained(args.model,local_files_only=True,use_fast=True)
    if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
    rows=torch.load(args.tokens,map_location='cpu',weights_only=True)
    class Encoded(Dataset):
        def __len__(self): return len(rows)
        def __getitem__(self,i): return rows[i]
    class Collator:
        def __call__(self,features):
            m=max(len(x['input_ids']) for x in features); m=((m+7)//8)*8
            ids=[]; masks=[]; labels=[]
            for x in features:
                n=m-len(x['input_ids']); ids.append(x['input_ids']+[tokenizer.pad_token_id]*n); masks.append([1]*len(x['input_ids'])+[0]*n); labels.append(x['labels']+[-100]*n)
            labels_t=torch.tensor(labels,dtype=torch.long)
            if labels_t.shape[0] != 1: raise RuntimeError('Selective logits collator currently requires per-device batch size 1')
            # A label at position t is predicted by the logit at t-1; keep only those logits.
            shifted=labels_t[:,1:]; keep_mask=shifted.ne(-100)
            logits_to_keep=torch.where(keep_mask[0])[0]
            shift_labels=shifted[keep_mask]
            if logits_to_keep.numel() == 0: raise RuntimeError('Batch contains no supervised next-token targets')
            return {'input_ids':torch.tensor(ids,dtype=torch.long),'attention_mask':torch.tensor(masks,dtype=torch.long),'labels':labels_t,'logits_to_keep':logits_to_keep,'shift_labels':shift_labels}
    bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=dtype)
    model_kwargs={'quantization_config':bnb,'device_map':{'':rank},'local_files_only':True,'low_cpu_mem_usage':True,'attn_implementation':'sdpa'}
    import transformers
    model_kwargs['dtype' if int(transformers.__version__.split('.')[0])>=5 else 'torch_dtype']=dtype
    model=AutoModelForMultimodalLM.from_pretrained(args.model,**model_kwargs)
    model.config.use_cache=False
    model=prepare_model_for_kbit_training(model,use_gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False})
    cfg=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias='none',task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
    model=get_peft_model(model,cfg)
    if rank==0: model.print_trainable_parameters()
    model.train(); probe_i=max(range(len(rows)),key=lambda i:len(rows[i]['input_ids'])); probe=Collator()([rows[probe_i]])
    if rank==0: print(f"Preflighting longest training sequence: {len(rows[probe_i]['input_ids'])} tokens on {torch.cuda.get_device_name(rank)}",flush=True)
    probe={k:v.to(rank) for k,v in probe.items()}; loss=model(**probe).loss
    if not torch.isfinite(loss): raise RuntimeError('QLoRA forward/backward preflight returned non-finite loss')
    loss.backward(); model.zero_grad(set_to_none=True); del probe,loss; torch.cuda.empty_cache()
    if rank==0: print('QLoRA forward/backward preflight passed on all visible ranks',flush=True)
    out=Path(args.out); checkpoints=out/'checkpoints'; checkpoints.mkdir(parents=True,exist_ok=True)
    training_kwargs=dict(output_dir=str(checkpoints),num_train_epochs=args.epochs,per_device_train_batch_size=1,gradient_accumulation_steps=args.grad_accum,learning_rate=args.lr,weight_decay=0.01,warmup_steps=0.05,lr_scheduler_type='cosine',logging_steps=10,save_strategy='epoch',save_total_limit=3,optim='adamw_torch',bf16=dtype==torch.bfloat16,fp16=dtype==torch.float16,gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False},ddp_find_unused_parameters=False if world>1 else None,remove_unused_columns=False,report_to=[],seed=args.seed,data_seed=args.seed)
    trainer=Trainer(model=model,args=TrainingArguments(**training_kwargs),train_dataset=Encoded(),data_collator=Collator())
    result=trainer.train()
    adapter=out/'adapter_final'; trainer.save_model(str(adapter))
    if trainer.is_world_process_zero():
        tokenizer.save_pretrained(adapter)
        report={'epochs_requested':args.epochs,'learning_rate':args.lr,'world_size':world,'effective_batch_size':world*args.grad_accum,'max_seq_length':args.max_length,'train_rows':len(rows),'train_runtime_seconds':result.metrics.get('train_runtime'),'train_loss':result.metrics.get('train_loss'),'log_history':trainer.state.log_history}
        (out/'training_report.json').write_text(json.dumps(report,indent=2,ensure_ascii=False)+'\n',encoding='utf-8')
        print(json.dumps({k:v for k,v in report.items() if k!='log_history'},indent=2),flush=True)
        if not list(adapter.glob('adapter_model*.safetensors')): raise RuntimeError('Adapter weights were not saved')
        print('ADAPTER_READY',str(adapter),flush=True)
if __name__=='__main__': main()
'''
WORKER_PATH = OUTPUT_ROOT / 'train_qwen35_lora.py'
WORKER_PATH.write_text(WORKER_SCRIPT, encoding='utf-8')
grad_accum = max(1, (EFFECTIVE_BATCH_TARGET + GPU_COUNT - 1) // GPU_COUNT)
print(f"GPU_COUNT={GPU_COUNT}; grad_accum={grad_accum}; effective_batch={GPU_COUNT * grad_accum}")
command = [sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node', str(GPU_COUNT), str(WORKER_PATH), '--model', str(MODEL_DIR), '--tokens', str(TOKENIZED_PATH), '--out', str(OUTPUT_ROOT), '--epochs', str(NUM_EPOCHS), '--lr', str(LEARNING_RATE), '--max-length', str(MAX_SEQ_LENGTH), '--grad-accum', str(grad_accum), '--seed', str(SEED)]
print('Launching QLoRA:', ' '.join(command))
subprocess.run(command, check=True)
TOKENIZED_PATH.unlink(missing_ok=True)
print('Training complete. Adapter:', OUTPUT_ROOT / 'adapter_final')


## After training

The output folder contains the adapter, epoch checkpoints, and a training report. Download those artifacts from Kaggle. Keep the base model reference and its exact version with the adapter.

The notebook intentionally does not publish a kernel, create a Kaggle dataset, submit a competition entry, or overwrite any model. For Ollama, merge the adapter into the same Qwen3.5-4B base, convert the merged HF checkpoint to GGUF with a llama.cpp build that supports Qwen3.5, then quantize to the same GGUF level as the current Ollama model. Run the project’s official BTC val309 evaluation on that GGUF artifact before considering promotion.

If the training run fails before `ADAPTER_READY`, inspect the last traceback and Kaggle GPU/runtime limits; epoch checkpoints are preserved under `checkpoints/`.
